In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Use GPU automatically
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)  # Should print: cuda

Using: cuda


In [27]:
def load_image(path, max_dim=512):
    img = Image.open(path).convert('RGB')
    
    # Resize keeping proportions
    scale = max_dim / max(img.size)
    new_size = (int(img.size[0] * scale), int(img.size[1] * scale))
    img = img.resize(new_size, Image.LANCZOS)
    
    # Convert to tensor and normalize for VGG
    transform = transforms.Compose([
        transforms.ToTensor(),           # [0,255] → [0,1]
        transforms.Normalize(            # VGG19 normalization
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    img = transform(img).unsqueeze(0)   # add batch dim
    return img.to(device)               # send to GPU

content_image = load_image('input_path')
style_image   = load_image('output_path')

In [12]:
class StyleTransferModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Load pretrained VGG19
        vgg = models.vgg19(pretrained=True).features
        vgg.eval()
        
        # Freeze all weights
        for param in vgg.parameters():
            param.requires_grad = False
        
        # Split into blocks — same layers as TF version
        self.block1 = vgg[:2]   # block1_conv1
        self.block2 = vgg[2:7]  # block2_conv1
        self.block3 = vgg[7:12] # block3_conv1
        self.block4 = vgg[12:21]# block4_conv1
        self.block5 = vgg[21:30]# block5_conv1
        self.block5_conv2 = vgg[30:32] # block5_conv2 → content

    def forward(self, x):
        s1 = self.block1(x)
        s2 = self.block2(s1)
        s3 = self.block3(s2)
        s4 = self.block4(s3)
        s5 = self.block5(s4)
        content = self.block5_conv2(s5)
        
        style_features   = [s1, s2, s3, s4, s5]
        content_features = [content]
        return style_features, content_features

model = StyleTransferModel().to(device)

In [28]:
def gram_matrix(feature_map):
    b, C, H, W = feature_map.shape
    features = feature_map.view(C, H * W)        # reshape
    gram = torch.mm(features, features.t())       # matmul
    return gram / (C * H * W)                     # normalize

In [29]:
def content_loss(content_features, generated_features):
    loss = 0
    for cf, gf in zip(content_features, generated_features):
        loss += torch.mean((cf - gf) ** 2)
    return loss

def style_loss(style_features, generated_features):
    loss = 0
    for sf, gf in zip(style_features, generated_features):
        gram_style     = gram_matrix(sf)
        gram_generated = gram_matrix(gf)
        loss += torch.mean((gram_style - gram_generated) ** 2)
    return loss

In [42]:
# Start from content image
generated_image = content_image.clone().requires_grad_(True)

optimizer = torch.optim.Adam([generated_image], lr=0.02)

# Pre-compute targets once
with torch.no_grad():
    style_targets,   _ = model(style_image)
    _,  content_targets = model(content_image)

alpha = 1e4
beta  = 1e7

for step in range(3000):
    optimizer.zero_grad()
    
    # Get features of generated image
    style_gen, content_gen = model(generated_image)
    
    # Compute losses
    c_loss = content_loss(content_targets, content_gen)
    s_loss = style_loss(style_targets, style_gen)
    total_loss = alpha * c_loss + beta * s_loss
    
    # Gradient descent on pixels
    total_loss.backward()
    optimizer.step()
    
    # Clip pixel values
    with torch.no_grad():
        generated_image.clamp_(0, 1)
    
    if step % 100 == 0:
        print(f"Step {step} | Loss: {total_loss.item():.2f}")

Step 0 | Loss: 17681.28
Step 100 | Loss: 9995.32
Step 200 | Loss: 6404.35
Step 300 | Loss: 5211.69
Step 400 | Loss: 4371.36
Step 500 | Loss: 3969.37
Step 600 | Loss: 3690.83
Step 700 | Loss: 3436.11
Step 800 | Loss: 3182.89
Step 900 | Loss: 3045.63
Step 1000 | Loss: 2913.64
Step 1100 | Loss: 2810.24
Step 1200 | Loss: 2781.65
Step 1300 | Loss: 2631.67
Step 1400 | Loss: 2545.38
Step 1500 | Loss: 2579.88
Step 1600 | Loss: 2467.87
Step 1700 | Loss: 2366.06
Step 1800 | Loss: 2325.68
Step 1900 | Loss: 2323.44
Step 2000 | Loss: 2269.57
Step 2100 | Loss: 2187.83
Step 2200 | Loss: 2138.93
Step 2300 | Loss: 2332.42
Step 2400 | Loss: 2198.06
Step 2500 | Loss: 2114.36
Step 2600 | Loss: 2079.74
Step 2700 | Loss: 2020.04
Step 2800 | Loss: 1991.58
Step 2900 | Loss: 1985.26


In [43]:
def save_image(tensor, filename):
    img = tensor.squeeze(0).detach().cpu()  # remove batch, move to CPU
    img = transforms.ToPILImage()(img)
    img.save(filename)
    img.show()

save_image(generated_image, 'save_path')